# Sesión 1 · Del dato a la pregunta
**Análisis Estadístico de la Información — Especialización en Big Data (UCompensar)**

Este cuaderno (*notebook*) acompaña las [diapositivas de la sesión](https://jotamao1985.github.io/ANaLISIS-ESTADiSTICO-DE-LA-INFORMACIoN_Ucomp/diapositivas/sesion-1-del-dato-a-la-pregunta.html).
Cada celda lleva entre corchetes el mismo identificador que el código proyectado, así que puedes seguir
la clase celda a celda. La portada del curso está en https://jotamao1985.github.io/ANaLISIS-ESTADiSTICO-DE-LA-INFORMACIoN_Ucomp/

1. **Archivo → Guardar una copia en Drive** para que tus cambios no se pierdan.
2. Ejecuta las celdas **en orden** con **Mayús + Enter**. Varias usan resultados de las anteriores.
   Si Colab avisa que el cuaderno no es de Google, elige **Ejecutar de todos modos**.
3. Si Colab se reinicia o una celda falla al repetirla, usa **Entorno de ejecución → Ejecutar anteriores**.
4. La primera celda crea los datos del caso *Tingua Domicilios S.A.S.*, una empresa **ficticia**:
   sus datos se inventan con una semilla fija y no corresponden a personas ni a empresas reales.

In [ ]:
#@title ▶ Crea los datos del caso (ejecútala sin modificar) { display-mode: "form" }
"""Datos del caso «Tingua Domicilios S.A.S.» (empresa FICTICIA).

Genera datos/pedidos.csv: los 12 000 pedidos que la plataforma entregó entre el
1 de enero y el 30 de junio de 2026 en cinco ciudades. Todo se inventa con una
semilla fija: los datos no corresponden a personas ni a empresas reales.

Solo usa la biblioteca estándar y, del generador aleatorio, solo random(), que
es lo único que Python garantiza igual entre versiones. Por eso la tabla sale
idéntica en Colab y en cualquier computador.

El caso sirve para todo el curso: variables de todos los tipos (sesión 1),
asimetrías y algunos atípicos (medidas y datos atípicos), una población
completa de la que se pueden sacar muestras (muestreo) y un tiempo de entrega
que depende de la distancia, la lluvia y otras variables (regresión).
"""

import csv
import math
import random
from datetime import date, timedelta
from pathlib import Path

SEMILLA = 2026
N_PEDIDOS = 12_000
INICIO = date(2026, 1, 1)
DIAS = 181  # del 1 de enero al 30 de junio de 2026

COLUMNAS = ["id_pedido", "fecha", "franja", "ciudad", "categoria", "n_productos",
            "valor_pedido", "medio_pago", "estrato", "suscriptor", "distancia_km",
            "lluvia", "temperatura_c", "tiempo_entrega_min", "calificacion"]

# ciudad: (peso, temperatura media °C, desviación, prob. de lluvia, minutos extra, factor de distancia)
CIUDADES = {
    "Bogotá":       (0.40, 14.5, 2.2, 0.30, 5.0, 1.15),
    "Medellín":     (0.22, 22.5, 2.3, 0.28, 2.0, 1.00),
    "Cali":         (0.16, 25.0, 2.3, 0.22, 0.0, 1.00),
    "Barranquilla": (0.12, 29.0, 1.8, 0.12, -1.0, 0.95),
    "Bucaramanga":  (0.10, 23.5, 2.2, 0.20, -2.0, 0.90),
}
LLUVIA_MES = {1: 0.6, 2: 0.7, 3: 1.0, 4: 1.4, 5: 1.3, 6: 0.8}

# franja: (peso, ajuste de temperatura °C, minutos extra)
FRANJAS = {
    "Mañana":   (0.15, -1.5, 0.0),
    "Almuerzo": (0.35, 1.5, 4.0),
    "Tarde":    (0.20, 1.0, 0.0),
    "Noche":    (0.30, -2.0, 3.0),
}

# categoría: (productos extra en promedio, precio medio por producto en COP, minutos de preparación)
CATEGORIAS = {
    "Restaurante": (1.3, 27_000, 3.0),
    "Mercado":     (6.0, 8_500, 4.0),
    "Farmacia":    (0.9, 21_000, 0.0),
    "Mascotas":    (1.2, 36_000, 1.0),
    "Floristería": (0.2, 88_000, 2.0),
}
# Qué se pide en cada franja (mismo orden que CATEGORIAS)
CATEGORIA_POR_FRANJA = {
    "Mañana":   (0.30, 0.35, 0.20, 0.10, 0.05),
    "Almuerzo": (0.78, 0.08, 0.07, 0.04, 0.03),
    "Tarde":    (0.40, 0.25, 0.15, 0.12, 0.08),
    "Noche":    (0.70, 0.10, 0.12, 0.05, 0.03),
}
ESTRATOS = {1: 0.06, 2: 0.22, 3: 0.33, 4: 0.20, 5: 0.11, 6: 0.08}


class _Azar:
    """Lo que hace falta del azar, construido solo sobre random()."""

    def __init__(self, semilla):
        self.r = random.Random(semilla)

    def u(self):
        return self.r.random()

    def elegir(self, opciones, pesos):
        x, acum = self.u() * sum(pesos), 0.0
        for opcion, peso in zip(opciones, pesos):
            acum += peso
            if x < acum:
                return opcion
        return opciones[-1]

    def normal(self, media=0.0, desv=1.0):
        # Box-Muller
        u1, u2 = 1.0 - self.u(), self.u()
        return media + desv * math.sqrt(-2.0 * math.log(u1)) * math.cos(2.0 * math.pi * u2)

    def exponencial(self, media):
        return -media * math.log(1.0 - self.u())

    def poisson(self, lam):
        # Knuth: suficiente para medias pequeñas
        limite, k, p = math.exp(-lam), 0, 1.0
        while True:
            p *= self.u()
            if p <= limite:
                return k
            k += 1


def _redondear(x, decimales=0):
    factor = 10 ** decimales
    return math.floor(x * factor + 0.5) / factor


def _pedido(az, dias, pesos_dias):
    fecha = az.elegir(dias, pesos_dias)
    ciudad = az.elegir(list(CIUDADES), [v[0] for v in CIUDADES.values()])
    _, t_media, t_desv, p_lluvia, min_ciudad, f_dist = CIUDADES[ciudad]
    franja = az.elegir(list(FRANJAS), [v[0] for v in FRANJAS.values()])
    _, t_franja, min_franja = FRANJAS[franja]
    categoria = az.elegir(list(CATEGORIAS), CATEGORIA_POR_FRANJA[franja])
    extra, precio, min_prep = CATEGORIAS[categoria]
    estrato = az.elegir(list(ESTRATOS), list(ESTRATOS.values()))

    suscriptor = az.u() < 0.10 + 0.06 * (estrato - 1)
    p_efectivo = 0.45 - 0.07 * (estrato - 1)
    medio_pago = az.elegir(["Efectivo", "Billetera digital", "Tarjeta"],
                           [p_efectivo, 0.25, 1 - p_efectivo - 0.25])

    n = 1 + az.poisson(extra)
    if categoria in ("Mercado", "Restaurante") and az.u() < 0.003:
        n = 20 + int(az.u() * 26)  # pedido corporativo: pocos, pero existen
    valor = n * precio * math.exp(az.normal(0, 0.22)) * (1 + 0.07 * (estrato - 3))
    valor = max(3_000, _redondear(valor / 100) * 100)

    distancia = math.exp(az.normal(math.log(3.0), 0.5)) * f_dist
    distancia = min(14.0, max(0.4, _redondear(distancia, 1)))

    lluvia = az.u() < p_lluvia * LLUVIA_MES[fecha.month]
    temperatura = _redondear(az.normal(t_media + t_franja - 1.8 * lluvia, t_desv), 1)

    tiempo = (11 + 3.1 * distancia + 0.45 * min(n, 15) + min_prep + min_franja + min_ciudad
              + 7.5 * lluvia - 2.5 * suscriptor + az.normal(0, 4) + az.exponencial(3))
    if az.u() < 0.005:
        tiempo += 55 + 95 * az.u()  # pedidos que se demoraron muchísimo
    tiempo = max(6.0, _redondear(tiempo, 1))

    latente = 6.4 - 0.065 * tiempo + 0.2 * suscriptor + az.normal(0, 0.6)
    estrellas = int(min(5, max(1, math.floor(latente + 0.5))))
    califico = az.u() < (0.80 if tiempo > 55 else 0.58)

    return {
        "fecha": fecha, "franja": franja, "ciudad": ciudad, "categoria": categoria,
        "n_productos": n, "valor_pedido": int(valor), "medio_pago": medio_pago,
        "estrato": estrato, "suscriptor": "Sí" if suscriptor else "No",
        "distancia_km": distancia, "lluvia": "Sí" if lluvia else "No",
        "temperatura_c": temperatura, "tiempo_entrega_min": tiempo,
        "calificacion": estrellas if califico else "",
    }


def crear_datos(carpeta="datos"):
    """Escribe <carpeta>/pedidos.csv y devuelve su ruta."""
    az = _Azar(SEMILLA)
    dias = [INICIO + timedelta(days=d) for d in range(DIAS)]
    pesos_dias = [1.35 if d.weekday() >= 4 else 1.0 for d in dias]  # viernes a domingo, más pedidos
    pedidos = [_pedido(az, dias, pesos_dias) for _ in range(N_PEDIDOS)]
    orden_franja = list(FRANJAS)
    pedidos.sort(key=lambda p: (p["fecha"], orden_franja.index(p["franja"])))

    ruta = Path(carpeta) / "pedidos.csv"
    ruta.parent.mkdir(parents=True, exist_ok=True)
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        escritor = csv.DictWriter(f, fieldnames=COLUMNAS)
        escritor.writeheader()
        for i, p in enumerate(pedidos, start=1):
            escritor.writerow({"id_pedido": f"P{i:05d}", **p, "fecha": p["fecha"].isoformat()})
    miles = f"{N_PEDIDOS:,}".replace(",", "\u202f")  # 12 000, con espacio fino
    print(f"Listo: {ruta} con {miles} pedidos y {len(COLUMNAS)} columnas")
    return ruta


"""Ayudas comunes a todos los notebooks del curso (van en la celda oculta, tras genera_datos.py).

Las de cada sesión están en ayudas_sesionN.py.
"""


def _aviso_version():
    """Las diapositivas se ejecutaron con pandas 2.2; con pandas 3 algunas salidas cambian."""
    import pandas as pd

    if int(pd.__version__.split(".")[0]) >= 3:
        print(f"Aviso: este Colab trae pandas {pd.__version__} y las diapositivas usan pandas 2.2. "
              "Algunas salidas se verán un poco distintas (por ejemplo, el texto sale como str "
              "y no como object).")


_aviso_version()


"""Verificadores de la práctica de la sesión 1: dan pistas sin revelar la respuesta.

Van dentro de la celda oculta del notebook, junto con genera_datos.py y ayudas.py.
"""

import unicodedata


# Formas equivalentes que también se aceptan
_SINONIMOS = {
    "cualitativa nominal": "nominal", "cualitativa ordinal": "ordinal",
    "cuantitativa discreta": "discreta", "cuantitativa continua": "continua",
    "de razon": "razon", "proporcion": "razon", "de proporcion": "razon",
    "ratio": "razon", "de intervalo": "intervalo", "intervalar": "intervalo",
}


def _normal(texto):
    texto = unicodedata.normalize("NFD", " ".join(str(texto).lower().split()))
    texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
    return _SINONIMOS.get(texto, texto)


_TIPOS = {
    "franja": ("ordinal", "ordinal"),
    "ciudad": ("nominal", "nominal"),
    "categoria": ("nominal", "nominal"),
    "n_productos": ("discreta", "razon"),
    "valor_pedido": ("continua", "razon"),
    "medio_pago": ("nominal", "nominal"),
    "estrato": ("ordinal", "ordinal"),
    "suscriptor": ("nominal", "nominal"),
    "distancia_km": ("continua", "razon"),
    "lluvia": ("nominal", "nominal"),
    "temperatura_c": ("continua", "intervalo"),
    "tiempo_entrega_min": ("continua", "razon"),
    "calificacion": ("ordinal", "ordinal"),
}
_CUALI, _CUANTI = {"nominal", "ordinal"}, {"discreta", "continua"}
_ESCALAS = {"nominal", "ordinal", "intervalo", "razon"}


def _pista_tipo(dado, correcto):
    if dado not in _CUALI | _CUANTI:
        return "el tipo debe ser nominal, ordinal, discreta o continua"
    if (dado in _CUALI) != (correcto in _CUALI):
        return "¿tiene sentido sumar o promediar sus valores?"
    if correcto in _CUALI:
        return "¿sus categorías siguen un orden o una secuencia natural, o solo son nombres?"
    return "¿se cuenta (0, 1, 2…) o se mide (cualquier valor dentro de un rango)?"


def _pista_escala(dado, correcto):
    if dado not in _ESCALAS:
        return "la escala debe ser nominal, ordinal, intervalo o razón"
    if {dado, correcto} == {"intervalo", "razon"}:
        return "¿el cero significa «nada» de esa cantidad?"
    return "la escala tiene que ser coherente con el tipo"


def revisar_tipos(mis_tipos):
    """Compara tu clasificación con la esperada y da una pista en cada error."""
    if not isinstance(mis_tipos, dict):
        print('✗  mis_tipos debe ser un diccionario: {"franja": ("tipo", "escala"), ...}')
        return
    tuyas = {_normal(k): v for k, v in mis_tipos.items()}  # «categoría» = «categoria»
    aciertos = 0
    for variable, (tipo_ok, escala_ok) in _TIPOS.items():
        if variable not in tuyas:
            print(f"·  {variable}: falta en tu diccionario")
            continue
        par = tuyas[variable]
        if not isinstance(par, (tuple, list)) or len(par) != 2:
            print(f'✗  {variable}: escribe una pareja ("tipo", "escala"), '
                  "con una coma entre los dos textos")
            continue
        tipo, escala = (_normal(x) for x in par)
        if tipo == "?" or escala == "?":
            print(f"·  {variable}: todavía tiene «?»")
            continue
        if variable == "valor_pedido" and tipo == "discreta":
            tipo = "continua"  # en rigor se cuenta en pesos; se trata como continua
        pistas = []
        if variable == "calificacion" and tipo in _CUANTI:
            pistas.append("las apps promedian las estrellas, pero ¿hay la misma distancia entre 1 y 2 "
                          "estrellas que entre 4 y 5? ¿Qué otra variable de hoy usa números solo "
                          "para ordenar?")
        elif tipo != tipo_ok:
            pistas.append(_pista_tipo(tipo, tipo_ok))  # con el tipo mal, la escala se corrige sola
        elif escala != escala_ok:
            pistas.append(_pista_escala(escala, escala_ok))
        if pistas:
            print(f"✗  {variable}: " + " · ".join(pistas))
        else:
            aciertos += 1
            print(f"✓  {variable}")
    print(f"\n{aciertos} de {len(_TIPOS)} variables bien clasificadas")


def revisar_escalas(tabla):
    """Revisa que franja y calificacion sean categorías ordenadas y fecha, una fecha."""
    import pandas as pd

    if not isinstance(tabla, pd.DataFrame):
        print("✗  pásale la tabla: revisar_escalas(pedidos)")
        return
    faltan = [c for c in ("franja", "calificacion", "fecha") if c not in tabla.columns]
    if faltan:
        print(f"✗  a la tabla le falta: {', '.join(faltan)}; vuelve a ejecutar [cargar]")
        return
    original = pd.read_csv("datos/pedidos.csv")

    def ordenada(columna, categorias):
        serie = tabla[columna]
        if isinstance(serie, pd.DataFrame):
            return f"✗  {columna}: la tabla tiene dos columnas {columna}; vuelve a ejecutar [cargar]"
        if not isinstance(serie.dtype, pd.CategoricalDtype):
            return f"✗  {columna}: todavía es {serie.dtype}; conviértela con pd.Categorical"
        if not serie.cat.ordered:
            return f"✗  {columna}: es categoría, pero falta ordered=True"
        texto = lambda c: str(c).removesuffix(".0")  # 4 y 4.0 son la misma estrella
        tuyas = [texto(c) for c in serie.cat.categories]
        esperadas = [texto(c) for c in categorias]
        if sorted(tuyas) != sorted(esperadas):
            return (f"✗  {columna}: revisa las categorías: deben escribirse igual que "
                    "en los datos (mayúsculas, tildes, sin espacios de más)")
        if tuyas != esperadas:
            return f"✗  {columna}: las categorías están bien escritas; revisa su orden"
        if serie.notna().sum() != original[columna].notna().sum():
            return (f"✗  {columna}: se perdieron valores al convertir; "
                    "revisa que las categorías se escriban igual que en los datos")
        return f"✓  {columna}: categoría ordenada"

    print(ordenada("franja", ["Mañana", "Almuerzo", "Tarde", "Noche"]))
    print(ordenada("calificacion", [1, 2, 3, 4, 5]))
    if pd.api.types.is_datetime64_any_dtype(tabla["fecha"]) and (
            tabla["fecha"].notna().sum() < original["fecha"].notna().sum()):
        print("✗  fecha: es fecha, pero se perdieron valores; "
              "los datos vienen como 2026-01-31 (año-mes-día)")
    elif pd.api.types.is_datetime64_any_dtype(tabla["fecha"]):
        inicio, fin = tabla["fecha"].min().date(), tabla["fecha"].max().date()
        print(f"✓  fecha: es una fecha (del {inicio} al {fin})")
    else:
        print(f"✗  fecha: todavía es {tabla['fecha'].dtype}; conviértela con pd.to_datetime")

# Crea datos/pedidos.csv: los 12 000 pedidos del primer semestre de 2026
ruta = crear_datos("datos")

## 1 · Primer contacto con los datos

In [ ]:
# [cargar]
import pandas as pd

pedidos = pd.read_csv("datos/pedidos.csv")
pedidos.shape          # (filas, columnas)

In [ ]:
# [vistazo]
# Las primeras cinco filas: cada fila es un pedido, cada columna una variable
pedidos.head()

### Diccionario de datos

| Columna | Qué registra | Valores o unidad | Origen |
|---|---|---|---|
| `id_pedido` | Código del pedido | P00001, P00002… | App |
| `fecha` | Día del pedido | AAAA-MM-DD, del 1 de enero al 30 de junio de 2026 | App |
| `franja` | Momento del día en que se pidió | Mañana, Almuerzo, Tarde, Noche | App |
| `ciudad` | Ciudad de entrega | Bogotá, Medellín, Cali, Barranquilla, Bucaramanga | App |
| `categoria` | Tipo de comercio que despacha | Restaurante, Mercado, Farmacia, Mascotas, Floristería | App |
| `n_productos` | Unidades pedidas | conteo | App |
| `valor_pedido` | Valor de los productos, sin domicilio ni propina | pesos (COP) | App |
| `medio_pago` | Cómo se pagó | Efectivo, Tarjeta, Billetera digital | App |
| `estrato` | Estrato de la **dirección de entrega**: lo asigna el municipio a la vivienda; no es el ingreso de quien pide | 1 a 6 | Estratificación del municipio, unida por dirección |
| `suscriptor` | Si quien pide paga la suscripción de Tingua | Sí / No | App |
| `distancia_km` | Distancia por vía del comercio a la dirección | kilómetros; la app registra hasta 14: una ruta más larga queda como 14 | App |
| `lluvia` | Si llovía en la zona durante la entrega | Sí / No | Servicio externo de clima |
| `temperatura_c` | Temperatura en la zona al momento de la entrega | grados Celsius | Servicio externo de clima |
| `tiempo_entrega_min` | Minutos desde que se confirma el pedido hasta que llega (incluye la preparación) | minutos | App |
| `calificacion` | Estrellas que dio quien pidió; vacía si no calificó | 1 a 5 | App (calificar es opcional) |

In [ ]:
# [tipos]
# El tipo de dato que pandas adivinó para cada columna
pedidos.dtypes

In [ ]:
# [faltantes]
# calificacion es float64 aunque sean estrellas de 1 a 5: ¿por qué?
faltan = pedidos["calificacion"].isna()
print("Pedidos sin calificación:", faltan.sum())
print("Proporción:", round(faltan.mean(), 3))

## 2 · Declarar la escala de medición

pandas guarda el estrato como un número entero (`int64`), y con un número se puede promediar.
Pero el estrato es una **categoría con orden**: decírselo a pandas evita cálculos sin sentido.

In [ ]:
# [ordinal]
# El estrato es una categoría con orden, no una cantidad
pedidos["estrato"] = pd.Categorical(
    pedidos["estrato"], categories=[1, 2, 3, 4, 5, 6], ordered=True
)
# Comparar sí tiene sentido...
print("Estrato máximo:", pedidos["estrato"].max())
# ...promediar, no
try:
    pedidos["estrato"].mean()
except TypeError as error:
    print("TypeError:", error)

## 3 · Un parámetro y tres estadísticos

La tabla tiene **todos** los pedidos del semestre: es la población. Su media es un **parámetro** ($\mu$).
Una muestra aleatoria de 100 pedidos da un **estadístico** ($\bar{x}$), que cambia de muestra en muestra.

In [ ]:
# [parametro]
# Parámetro: se calcula con TODA la población
mu = pedidos["tiempo_entrega_min"].mean()
print(f"μ = {mu:.1f} min   (N = {len(pedidos)})")

# Estadísticos: cada muestra aleatoria da un valor distinto
for semilla in [1, 2, 3]:
    muestra = pedidos.sample(n=100, random_state=semilla)
    x_barra = muestra["tiempo_entrega_min"].mean()
    print(f"x̄ = {x_barra:.1f} min   (n = 100, semilla {semilla})")

## 4 · Un adelanto: contar categorías

Contar cuántas veces aparece cada categoría es el primer paso de una tabla de frecuencias,
el tema de la próxima sesión.

In [ ]:
# [frecuencias]
ciudad = pedidos["ciudad"]
conteo = ciudad.value_counts()
porcentaje = (ciudad.value_counts(normalize=True) * 100).round(1)
pd.DataFrame({"pedidos": conteo, "%": porcentaje})

---
# Práctica guiada

Cuatro ejercicios. Los dos primeros se revisan solos al ejecutar su celda de revisión;
el tercero lo comparamos en el chat y el cuarto lo comentamos en clase.

## Ejercicio 1 · Clasifica las variables

Para cada variable escribe su **tipo** (`nominal`, `ordinal`, `discreta` o `continua`) y su
**escala de medición** (`nominal`, `ordinal`, `intervalo` o `razón`; con o sin tilde). `id_pedido` y `fecha` no
están: el primero es un identificador y la segunda la trabajamos en el ejercicio 2.

`revisar_tipos` te dice cuáles están bien y, en las demás, te da una pista sin darte la respuesta.

In [ ]:
# [ej1]
mis_tipos = {
    # variable              (tipo,  escala)
    "franja":              ("?", "?"),
    "ciudad":              ("?", "?"),
    "categoria":           ("?", "?"),
    "n_productos":         ("?", "?"),
    "valor_pedido":        ("?", "?"),
    "medio_pago":          ("?", "?"),
    "estrato":             ("ordinal", "ordinal"),   # este ya lo hicimos juntos
    "suscriptor":          ("?", "?"),
    "distancia_km":        ("?", "?"),
    "lluvia":              ("?", "?"),
    "temperatura_c":       ("?", "?"),
    "tiempo_entrega_min":  ("?", "?"),
    "calificacion":        ("?", "?"),
}
revisar_tipos(mis_tipos)

## Ejercicio 2 · Declara las escalas

Como hicimos con el estrato:

- convierte `franja` en categoría ordenada (`Mañana` < `Almuerzo` < `Tarde` < `Noche`);
- convierte `calificacion` en categoría ordenada de 1 a 5 estrellas;
- convierte `fecha` en fecha con `pd.to_datetime`.

Reemplaza cada `...` y ejecuta. Después corre la celda de revisión. Como cada conversión parte
del archivo original, puedes repetir la celda las veces que quieras sin perder datos.

In [ ]:
# [ej2]
# Si ves «Ellipsis» en un error, te quedó un ... sin reemplazar
original = pd.read_csv("datos/pedidos.csv")

pedidos["franja"] = pd.Categorical(
    original["franja"], categories=..., ordered=True
)
pedidos["calificacion"] = pd.Categorical(
    original["calificacion"], categories=..., ordered=True
)
pedidos["fecha"] = pd.to_datetime(...)   # ← la columna "fecha" de original

In [ ]:
# [ej2_revision]
revisar_escalas(pedidos)

## Ejercicio 3 · Tu propio estadístico

La proporción de pedidos que se entregaron con lluvia, calculada con **toda** la población, es el
parámetro $p$. Toma tu propia muestra de 200 pedidos y calcula su proporción, $\hat{p}$.

Usa como semilla **un número de tres cifras que elijas tú** (del 100 al 999) y comparte tu $\hat{p}$
en el chat. ¿Por qué cada uno obtiene un valor distinto?

In [ ]:
# [ej3]
p = (pedidos["lluvia"] == "Sí").mean()
print(f"p = {p:.3f}   (los {len(pedidos)} pedidos)")

mi_semilla = 123   # ← un número de tres cifras que elijas tú
muestra = pedidos.sample(n=200, random_state=mi_semilla)
p_gorro = (muestra["lluvia"] == "Sí").mean()
print(f"p̂ = {p_gorro:.3f}   (n = 200, semilla {mi_semilla})")

## Ejercicio 4 · De los datos a la pregunta

Escribe **dos preguntas** que la gerencia de Tingua podría responder con estos datos. Para cada una:

1. ¿Qué variables necesitas y de qué tipo son?
2. ¿Basta con describir los 12 000 pedidos, o tendrías que inferir algo que no está en la tabla?

Y una pregunta sobre la fuente: ¿`pedidos.csv` es una fuente primaria o secundaria, interna o externa?
¿Y las columnas `lluvia` y `temperatura_c`, de dónde vienen? ¿Sus datos son observacionales o
experimentales? ¿Qué limita eso en lo que puedes concluir?

### Tus respuestas

*Doble clic para escribir aquí.*